## P96-PRUNE-CEO-DELETE-NOTEBOOK-R90: Lakelink Fiber PRI repo prune (Batch A + Batch B)
The repo has **237 objects excluding .git** (R93 scan, 2026-11-04, including this notebook; +1 vs R91: tests/test_mrc_format_dw130.py, which is not in a merge group). The cap is 200. The scheduled-agent safety filter refuses deletes, so **a human must run this notebook interactively**.

* **Cell 2 (dry run):** lists each target with its path, size and reason. It deletes nothing.
* **Cell 3 (Batch A, 20 objects):** removes `.ruff_cache/` (15), the stray `esbuild-0.1.2-py3-none-any.whl`, the empty `/src`, `/data` and `/resources` dirs, and the superseded `bundle/databricks.yml` stub (P21).
* **Cell 4 (Batch B gateway config):** prints the diff of `ai_gateway_config.json` against `ai-gateway-config.json`, then deletes the smaller underscore copy if no file references it (none did at R91). Review the diff first: if the underscore copy holds settings you need, merge them into the hyphen copy before you confirm.
* **Cell 5 (Batch B tests, then final recount):** merges `test_topology_dw*` (14 files), `test_scorecard_dw*` (3), `test_provisional_dw*` (2) and `test_conduits_dw*` (3) into one `*_dw_merged.py` per area. Each group is first built in a `/tmp` sandbox, and the full pytest suite runs before and after. A group is applied to the repo **only if every test outcome is identical** (same tests, same PASS/FAIL/SKIP). Otherwise the group is skipped and reported. The cell ends by recounting objects excluding .git and asserting that the total is under 200.

Set the `CONFIRM` widget to `YES` before running cells 3 to 5. With any other value, they only print what they would do.

Expected result (R93 refresh): 237 - 20 (A) - 1 (gateway) - 18 (tests, if all groups pass) = **198**. Group sizes were re-verified at R93: topology 14, scorecard 3, provisional 2, conduits 3. Cell 5 globs by prefix, so new files in those groups are picked up automatically. Cell 2 recomputes the real numbers, because agents keep adding `test_*_dwNNN.py` files.

In [0]:
import os, re, glob, sys, shutil, subprocess, difflib
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
dbutils.widgets.dropdown("CONFIRM", "NO", ["NO", "YES"])
WS = "/Users/stephen.hage@databricks.com/provable-resilience-intelligence"
REPO = "/Workspace" + WS
GROUPS = {"test_topology_dw": "test_topology_dw_merged.py",
          "test_scorecard_dw": "test_scorecard_dw_merged.py",
          "test_provisional_dw": "test_provisional_dw_merged.py",
          "test_conduits_dw": "test_conduits_dw_merged.py"}

def inventory():
    objs = []
    def walk(p):
        for o in w.workspace.list(p):
            if o.path == WS + "/.git" or o.path.startswith(WS + "/.git/"):
                continue  # note: /.github is kept
            objs.append(o)
            if o.object_type and o.object_type.value in ("DIRECTORY", "REPO"):
                walk(o.path)
    walk(WS)
    return objs

OBJS = inventory()
print("objects excl .git:", len(OBJS))
size = {o.path: (o.size or 0) for o in OBJS}
ruff = sorted(o.path for o in OBJS if o.path.startswith(WS + "/.ruff_cache"))
empty_dirs = [WS + d for d in ("/src", "/data", "/resources")
              if any(o.path == WS + d for o in OBJS)
              and not any(x.path.startswith(WS + d + "/") for x in OBJS)]
stub = WS + "/bundle/databricks.yml"
assert open("/Workspace" + stub).read().startswith("# SUPERSEDED"), "stub changed - re-check before deleting"
BATCH_A = [(WS + "/.ruff_cache", "regenerable lint cache (dir + contents)", len(ruff)),
           (WS + "/esbuild-0.1.2-py3-none-any.whl", "stray wheel, unreferenced", 1)] + \
          [(d, "empty directory", 1) for d in empty_dirs] + \
          [(stub, "superseded bundle stub (P21); root databricks.yml is active", 1)]
print("\nBATCH A (delete):")
for p, why, n in BATCH_A:
    print(f"  {p[len(WS):]:<45} {size.get(p, 0):>8} B  {n:>2} obj  {why}")
print("  total:", sum(n for *_, n in BATCH_A))
print("\nBATCH B (consolidate tests):")
for pre, tgt in GROUPS.items():
    fs = sorted(glob.glob(f"{REPO}/tests/{pre}[0-9]*.py"))
    print(f"  {pre}* : {len(fs)} files -> tests/{tgt}  (-{max(len(fs)-1, 0)})")
    for f in fs:
        print(f"      {os.path.basename(f):<32} {os.path.getsize(f):>7} B")
print("\nBATCH B (gateway config): delete /ai_gateway_config.json (duplicate of /ai-gateway-config.json)")

In [0]:
CONFIRM = dbutils.widgets.get("CONFIRM") == "YES"
for p, why, n in BATCH_A:
    if not CONFIRM:
        print("[dry-run] would delete", p[len(WS):]); continue
    try:
        w.workspace.delete(p, recursive=True)
        print("deleted", p[len(WS):], f"({n} obj)")
    except Exception as e:
        print("SKIP", p[len(WS):], "->", str(e)[:120])
# Stop the lint cache from coming back into the repo
gi = REPO + "/.gitignore"
if CONFIRM and ".ruff_cache" not in open(gi).read():
    with open(gi, "a") as f:
        f.write("\n.ruff_cache/\n")
    print("appended .ruff_cache/ to .gitignore")

In [0]:
keep, drop = REPO + "/ai-gateway-config.json", REPO + "/ai_gateway_config.json"
print("".join(difflib.unified_diff(open(drop).read().splitlines(True), open(keep).read().splitlines(True),
                                   "ai_gateway_config.json (drop)", "ai-gateway-config.json (keep)")))
refs = []
for dp, dn, fn in os.walk(REPO):
    if os.path.relpath(dp, REPO).split(os.sep)[0] in (".git", ".ruff_cache") or "node_modules" in dp:
        dn[:] = []; continue
    for f in fn:
        p = os.path.join(dp, f)
        if p == drop:
            continue
        try:
            if "ai_gateway_config.json" in open(p, errors="ignore").read():
                refs.append(p[len(REPO):])
        except OSError:
            pass
print("references to ai_gateway_config.json:", refs or "none")
if dbutils.widgets.get("CONFIRM") == "YES" and not refs:
    w.workspace.delete(WS + "/ai_gateway_config.json"); print("deleted /ai_gateway_config.json")
else:
    print("[dry-run or still referenced] not deleted")

In [0]:
LOADER = '''
# Auto-consolidated by @repo-pruner (P96). Each former test_<area>_dwNNN.py is kept verbatim in
# _SOURCES and executed in its own module namespace (no shared globals). It is registered in
# sys.modules under its old name, so cross-imports keep working. Tests are re-exported as
# test_dwNNN__<name>. To add a new DW regression, add a _SOURCES entry here instead of a new file.
import os as _os, sys as _sys, types as _types
_HERE = _os.path.dirname(_os.path.abspath(__file__))


def _load(_name, _src):
    if _name in _sys.modules:
        return _sys.modules[_name]
    _m = _types.ModuleType(_name)
    _m.__file__ = _os.path.join(_HERE, _name + ".py")
    _sys.modules[_name] = _m
    exec(compile(_src, _m.__file__, "exec"), _m.__dict__)
    return _m


def _export(_name, _m, _g):
    _tag = _name.rsplit("_", 1)[-1]
    if getattr(_m, "pytestmark", None) is not None:
        raise RuntimeError("module-level pytestmark not supported: " + _name)
    for _k, _v in list(vars(_m).items()):
        if hasattr(_v, "_pytestfixturefunction") or type(_v).__name__ == "FixtureFunctionDefinition":
            if _k in _g and _g[_k] is not _v:
                raise RuntimeError("fixture name collision: " + _k)
            _g[_k] = _v
        elif _k.startswith("test") and callable(_v) and getattr(_v, "__module__", None) == _name:
            _g["test_" + _tag + "__" + _k[4:].lstrip("_")] = _v
        elif _k.startswith("Test") and isinstance(_v, type) and _v.__module__ == _name:
            _g["Test" + _tag.upper() + "_" + _k[4:]] = _v


for _name, _src in _SOURCES.items():
    _export(_name, _load(_name, _src), globals())
'''
# test_topology_dw117 loads dw115 by file path. Route it through sys.modules instead.
DW117_OLD = '''    spec = importlib.util.spec_from_file_location(
        "_dw115_for_dw117", os.path.join(HERE, "test_topology_dw115.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod'''
DW117_NEW = '''    import sys as _s  # consolidated: dw115 is loaded earlier from _SOURCES
    return _s.modules["test_topology_dw115"]'''
Q3 = "'" * 3

def build_merged(tests_dir, prefix):
    files = sorted(glob.glob(os.path.join(tests_dir, prefix + "[0-9]*.py")))
    out = ['"""Consolidated ' + prefix + 'NNN regression tests (P96). Sources: '
           + ", ".join(os.path.basename(f) for f in files) + '."""', "_SOURCES = {}"]
    for p in files:
        name = os.path.basename(p)[:-3]
        src = open(p).read().replace(DW117_OLD, DW117_NEW)
        assert Q3 not in src and not src.rstrip().endswith("\\"), "cannot embed " + name
        out.append("_SOURCES[" + repr(name) + "] = r" + Q3 + src + Q3)
    return files, "\n".join(out) + "\n" + LOADER

def sandbox(dst):
    """Copy the code tree to /tmp, without .git, caches, node_modules or MLflow experiment refs."""
    shutil.rmtree(dst, ignore_errors=True)
    for dp, dn, fn in os.walk(REPO):
        rel = os.path.relpath(dp, REPO)
        if rel.split(os.sep)[0] in (".git", ".ruff_cache") or "node_modules" in dp or "__pycache__" in dp:
            dn[:] = []; continue
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        for f in fn:
            try:
                shutil.copy2(os.path.join(dp, f), os.path.join(dst, rel, f))
            except OSError:
                pass  # notebooks and experiments that WSFS cannot copy as plain files

def outcomes(d):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "-rA", "--tb=no",
                        "--continue-on-collection-errors", "tests/"], capture_output=True, text=True, cwd=d,
                       env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
    res = {}
    for l in re.sub(r"\x1b\[[0-9;]*m", "", r.stdout).splitlines():
        m = re.match(r"(PASSED|FAILED|SKIPPED|ERROR|XFAIL|XPASS) (\S+)", l)
        if m:
            res[m.group(2)] = m.group(1)
    return res

def map_id(tid, pre):
    m = re.match(r"tests/(" + pre + r")(\d+)\.py::(.+)$", tid)
    if not m:
        return tid
    num, rest = m.group(2), m.group(3)
    if rest.startswith("Test"):
        return f"tests/{GROUPS[pre]}::TestDW{num}_{rest[4:]}"
    return f"tests/{GROUPS[pre]}::test_dw{num}__{rest[4:].lstrip('_')}"

CONFIRM = dbutils.widgets.get("CONFIRM") == "YES"
BASE_DIR = "/tmp/p96_base"
sandbox(BASE_DIR)
base = outcomes(BASE_DIR)
assert base, "baseline pytest run produced no results; check pytest is installed"
print("baseline test ids:", len(base))
APPLIED = {}
for pre, tgt in GROUPS.items():
    d = f"/tmp/p96_{pre}"
    shutil.rmtree(d, ignore_errors=True)
    shutil.copytree(BASE_DIR, d)
    files, content = build_merged(f"{d}/tests", pre)
    if len(files) < 2:
        print(pre, "- nothing to merge"); continue
    with open(f"{d}/tests/{tgt}", "w") as fh:
        fh.write(content)
    for f in files:
        os.remove(f)
    after = outcomes(d)
    shutil.rmtree(d, ignore_errors=True)
    exp = {map_id(k, pre): v for k, v in base.items()}
    diff = {k: (exp.get(k), after.get(k)) for k in set(exp) | set(after) if exp.get(k) != after.get(k)}
    if diff:
        print(f"SKIP {pre}: {len(diff)} outcome differences, e.g.", list(diff.items())[:5]); continue
    print(f"OK   {pre}: {len(files)} files -> {tgt}, all {sum(1 for k in exp if GROUPS[pre] in k)} outcomes identical")
    if not CONFIRM:
        print("     [dry-run] repo unchanged"); continue
    # Write the merged file into the repo first, then remove the originals (verified in the sandbox above).
    files_repo, content_repo = build_merged(f"{REPO}/tests", pre)
    assert [os.path.basename(f) for f in files_repo] == [os.path.basename(f) for f in files], "repo changed since sandbox"
    with open(f"{REPO}/tests/{tgt}", "w") as fh:
        fh.write(content_repo)
    for f in files_repo:
        w.workspace.delete(WS + "/tests/" + os.path.basename(f))
    APPLIED[pre] = len(files_repo) - 1
shutil.rmtree(BASE_DIR, ignore_errors=True)
print("objects removed by test consolidation:", sum(APPLIED.values()), APPLIED)

# ---- Final recount (excl .git) and cap assertion ----
final = inventory()
print("objects excl .git: before", len(OBJS), "-> after", len(final))
if CONFIRM:
    assert len(final) < 200, f"still {len(final)} objects; review the SKIP lines above"
    print("PASS: under the 200 cap. @devops: commit the change. @repo-pruner: verify on the next run.")